In [ ]:
# Imports
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import rsatoolbox as rsa
from rsatoolbox.data import Dataset
from rsatoolbox.rdm import calc_rdm

from core import SantoroDataset, MODEL_CLASSES, extract_activations, load_yamnet_activations

# --- Config ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = "models"
RUNS = list(range(5))
MODEL_NAMES = ["waveform", "uninspired", "inspired"]
N_CLASSES = 50
BATCH_SIZE = 8

print(f"Device: {DEVICE}")
print(f"Model directory: {MODEL_DIR}")

Device: cpu
Model directory: models


In [7]:
# Load Data
print("Loading SantoroDataset...")
ds = SantoroDataset()
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
n_sounds = len(ds)
print(f"Number of sounds: {n_sounds}")

# The brain responses are in batch[2] of SantoroDataset
brain_activity = np.concatenate([batch[2].numpy() for batch in loader], axis=0)  # Shape: (288, n_voxels)

# Create an rsatoolbox Dataset. Each row is a measurement (a sound).
brain_data = Dataset(
    measurements=brain_activity,
    obs_descriptors={'condition': np.arange(n_sounds)}  # Each sound is its own condition
)

# Calculate the brain RDM using correlation distance
brain_rdm = calc_rdm(brain_data, method='correlation', descriptor='condition')
print("Brain RDM constructed.")
print(brain_rdm)

Loading SantoroDataset...
Number of sounds: 288
Brain RDM constructed.
rsatoolbox.rdm.RDMs
1 RDM(s) over 288 conditions

dissimilarity_measure = 
correlation

dissimilarities[0] = 
[[0.         0.41059334 0.38957562 ... 0.52725744 1.02617903 0.78970816]
 [0.41059334 0.         0.22283964 ... 0.58229168 1.01555804 0.83185841]
 [0.38957562 0.22283964 0.         ... 0.58473242 0.98956617 0.79421133]
 ...
 [0.52725744 0.58229168 0.58473242 ... 0.         1.01340008 0.5339717 ]
 [1.02617903 1.01555804 0.98956617 ... 1.01340008 0.         0.87632567]
 [0.78970816 0.83185841 0.79421133 ... 0.5339717  0.87632567 0.        ]]

descriptors: 

rdm_descriptors: 
index = [0]

pattern_descriptors: 
condition = [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(

In [ ]:
# Untrained Models
untrained_rdms = {}

# Rdm per model per layer
for name in MODEL_NAMES:
    print(f"Processing untrained: {name}")
    model = MODEL_CLASSES[name](num_classes=N_CLASSES).to(DEVICE).eval()
    
    with torch.no_grad():
        # extract_activations returns a dict: {layer_name: tensor (n_sounds, ...)}
        acts = extract_activations(loader, model)

    # Loop for each layer
    for layer_name, act in acts.items():
        # Flatten any spatial dimensions (e.g., for conv layers)
        act_np = act.numpy().reshape(act.shape[0], -1)  # Shape: (288, n_features)
        
        # Wrap in rsatoolbox Dataset and compute RDM
        d = Dataset(measurements=act_np, obs_descriptors={'condition': np.arange(n_sounds)})
        rdm = calc_rdm(d, method='correlation', descriptor='condition')
        
        key = f"{name}_untrained/{layer_name}"
        untrained_rdms[key] = rdm
        print(f"  -> {key}")

print(f"\nConstructed {len(untrained_rdms)} untrained model RDMs.")

Processing untrained: waveform
  -> waveform_untrained/hidden_layers.0
  -> waveform_untrained/hidden_layers.1
  -> waveform_untrained/hidden_layers.2
  -> waveform_untrained/hidden_layers.3
  -> waveform_untrained/hidden_layers.4
  -> waveform_untrained/classifier
Processing untrained: uninspired
  -> uninspired_untrained/conv_layers.0
  -> uninspired_untrained/conv_layers.1
  -> uninspired_untrained/conv_layers.2
  -> uninspired_untrained/fc_layers.0
  -> uninspired_untrained/fc_layers.1
  -> uninspired_untrained/classifier
Processing untrained: inspired
  -> inspired_untrained/conv_layers.0
  -> inspired_untrained/conv_layers.1
  -> inspired_untrained/conv_layers.2
  -> inspired_untrained/rnn
  -> inspired_untrained/classifier

Constructed 17 untrained model RDMs.


In [9]:
# Trained Models (5 Runs)
trained_rdms = {}

# Rdm per model per run per layer
for name in MODEL_NAMES:
    for run in RUNS:
        print(f"Processing trained: {name} run {run}")
        model = MODEL_CLASSES[name](num_classes=N_CLASSES)
        ckpt_path = os.path.join(MODEL_DIR, f"{name}_run{run}_best.pt")
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.to(DEVICE).eval()
        
        with torch.no_grad():
            acts = extract_activations(loader, model)
        
        for layer_name, act in acts.items():
            act_np = act.numpy().reshape(act.shape[0], -1)
            d = Dataset(measurements=act_np, obs_descriptors={'condition': np.arange(n_sounds)})
            rdm = calc_rdm(d, method='correlation', descriptor='condition')
            
            key = f"{name}_run{run}/{layer_name}"
            trained_rdms[key] = rdm

print(f"\nConstructed {len(trained_rdms)} trained model RDMs.")

Processing trained: waveform run 0
Processing trained: waveform run 1
Processing trained: waveform run 2
Processing trained: waveform run 3
Processing trained: waveform run 4
Processing trained: uninspired run 0
Processing trained: uninspired run 1
Processing trained: uninspired run 2
Processing trained: uninspired run 3
Processing trained: uninspired run 4
Processing trained: inspired run 0
Processing trained: inspired run 1
Processing trained: inspired run 2
Processing trained: inspired run 3
Processing trained: inspired run 4

Constructed 85 trained model RDMs.


In [10]:
# YAMNet rdms
yamnet_rdms = {}
yam_acts = load_yamnet_activations()  # Dict: {layer_name: (288, n_features)}

for layer_name, act in yam_acts.items():
    d = Dataset(measurements=act, obs_descriptors={'condition': np.arange(n_sounds)})
    rdm = calc_rdm(d, method='correlation', descriptor='condition')
    yamnet_rdms[f"yamnet/{layer_name}"] = rdm
    print(f"  -> yamnet/{layer_name}")

print(f"\nConstructed {len(yamnet_rdms)} YAMNet RDMs.")

  -> yamnet/layer01relu
  -> yamnet/layer02relu
  -> yamnet/layer03relu
  -> yamnet/layer04relu
  -> yamnet/layer05relu
  -> yamnet/layer06relu
  -> yamnet/layer07relu
  -> yamnet/layer08relu
  -> yamnet/layer09relu
  -> yamnet/layer10relu
  -> yamnet/layer11relu
  -> yamnet/layer12relu
  -> yamnet/layer13relu
  -> yamnet/layer14relu
  -> yamnet/embedding

Constructed 15 YAMNet RDMs.
